In [ ]:
!pip install -q einops
!wget -q https://huggingface.co/datasets/doantrongthai/mamba_package/resolve/main/pack_colab_torch2_14.zip -O /content/pack_colab_torch2_14.zip
!unzip -q -o /content/pack_colab_torch2_14.zip -d /

import torch
import selective_scan_cuda
from mamba_ssm import Mamba

layer = Mamba(d_model=64).cuda()
x = torch.randn(2, 16, 64, device="cuda")
out = layer(x)
print("SUCCESS:", out.shape)


In [ ]:
import os
if not os.getcwd().endswith("flood_mamba_edge"):
    if not os.path.exists("flood_mamba_edge"):
        !git clone https://github.com/doantrongthai/flood_mamba_edge.git
    %cd flood_mamba_edge
else:
    !git pull
!pip install -q -r requirements.txt
!pip install -q tabulate gdown pandas


In [ ]:
import os, shutil
try:
    from google.colab import files
except Exception:
    files = None

MODELS_TO_RUN = [
    "ablation_transconv_decoder",
    "ablation_triple_dilation",
    "ablation_nearest_decoder",
    "ablation_std_conv",
    "ablation_strip_horizontal",
    "ablation_strip_vertical",
    "ablation_single_stem",
    "ablation_no_bottleneck",
    "ablation_skip_add",
    "ablation_no_skip",
    "ablation_mamba_g4",
    "ablation_mamba_single_stage"
]

SEEDS_STR = "42 123 456 789 2024"
EPOCHS = 100
BATCH_SIZE = 4
IMAGE_SIZE = 128
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1

for idx, MODEL_NAME in enumerate(MODELS_TO_RUN, 1):
    print("\n" + "=" * 75)
    print(f"[{idx}/{len(MODELS_TO_RUN)}] RUNNING MODEL: {MODEL_NAME}")
    print("=" * 75 + "\n")
    
    # 1. Train 5 seeds
    !python main.py --dataset floodkaggle --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {SEEDS_STR} --download
    
    # 2. Cross Dataset Eval
    !python cross_dataset_eval.py --model {MODEL_NAME} --loss {LOSS_NAME} --source_dataset floodkaggle --target_datasets floodkaggle floodvn --seeds {SEEDS_STR} --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download
    
    # 3. Zip Artifacts and Download
    bundle_dir = f"{MODEL_NAME}_All_Artifacts"
    os.makedirs(bundle_dir, exist_ok=True)
    if os.path.exists("outputs"):
        for item in os.listdir("outputs"):
            s = os.path.join("outputs", item)
            d = os.path.join(bundle_dir, item)
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            elif os.path.isfile(s):
                shutil.copy2(s, d)
        zip_file = f"{MODEL_NAME}_artifacts.zip"
        shutil.make_archive(f"{MODEL_NAME}_artifacts", "zip", bundle_dir)
        print(f"[OK] Created archive: {zip_file}")
        if files is not None:
            try:
                files.download(zip_file)
            except Exception:
                pass
        shutil.rmtree("outputs")
        shutil.rmtree(bundle_dir)
        os.makedirs("outputs", exist_ok=True)
        print(f"[OK] Cleaned outputs for next model.\n")

print("\n[COMPLETED] ALL 12 MODELS FINISHED!")
